In [ ]:
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Read BBHI main data
m_df = pd.read_csv("~/Documents/2023:2024/Data/BBHI/BBHI Data Timept1 NPS.csv")

print(f"Number of participants: {len(m_df)}")

# Read BBHI education data
df_edu = pd.read_excel("~/Documents/2023:2024/Data/BBHI/BBHI_YoE.xlsx")

In [ ]:
# Calculate ages at baseline to filter df for 60+ yrs
def fix_year(date_str):
    """Fix two-digit year format to four-digit year format.
    
    Args:
        date_str (str): Date string in format mm/dd/yy or mm/dd/yyyy
    
    Returns:
        str: Date string in format mm/dd/yyyy
    """
    m, d, y = date_str.split("/")
    y = int(y)
    if y < 100:  # two digit year
        y += 2000 if y <= 25 else 1900 # otherwise it corrects all years to 2000
    return f"{m}/{d}/{y}"

def calculate_age(birth_date: str, collection_date: str) -> float:
    """Calculate exact age at baseline.

    Args:
        birth_date (str): Date of birth in format mm/dd/yyyy
        collection_date (str): Date of data collection in format mm/dd/yyyy

    Returns:
        age (float): Exact age at assessment
    """
    birth_date = datetime.strptime(fix_year(birth_date), "%m/%d/%Y")
    collection_date = datetime.strptime(fix_year(collection_date), "%m/%d/%Y")

    # Calculate difference in days
    days_difference = (collection_date - birth_date).days

    # Convert days to years using the average number of days in a year
    age = days_difference / 365.25
    return age

# 'Q1 GNRL Age' is DOB and 'w1_nps_date' is data collection date
# Add a new column to the DataFrame with exact age
m_df["w1_age"] = m_df.apply(lambda row: calculate_age(row["Q1 GNRL Age"], row["w1_nps_date"]), axis=1)

# Manually check that everything is running correctly
m_df[["id", "w1_age", "Q1 GNRL Age", "w1_nps_date"]].sample(10)

In [ ]:
# Filter df to only include 60+yrs
print(f"Number of participants before filtering: {len(m_df)}")
agefiltered_df = m_df[m_df["w1_age"] >= 60]
print(f"Number of participants after filtering: {len(agefiltered_df)}")

agefiltered_df[["id", "w1_age"]].head(5)

In [ ]:
# Merge dfs by participant ID for main data and education
filtered_df = pd.merge(agefiltered_df, df_edu, on="id", how="inner")
print(f"Number of participants: {len(filtered_df)}")

# Rename the ravlt variable
filtered_df.rename(columns={"w1_inm_recall_total_raw": "w1_ravlt_total"}, inplace=True)

In [ ]:
# Read BBHI timepoint 2 neuropsych data
np_tp2_df = pd.read_csv("~/Documents/2023:2024/Data/BBHI/BBHI Data Timept2 NPS.csv")

print(f"Number of participants: {len(np_tp2_df)}")

In [ ]:
# Rename tp2 variables
def add_prefix_to_selected_columns(df, columns_to_prefix, prefix="w2_"):
    """Add a prefix to specific column names in the DataFrame.

    Args:
        df (pd.DataFrame): The DataFrame where the column names will be updated.
        columns_to_prefix (list of str): The list of column names to which the prefix will be added.
        prefix (str): The prefix to add to each specified column name. Default is 'w2_'.

    Returns:
        pd.DataFrame: DataFrame with updated column names.
    """
    # Create a dictionary for renaming columns only in the specified list
    new_column_names = {col: prefix + col for col in df.columns if col in columns_to_prefix}
    df = df.rename(columns=new_column_names)
    return df

# List of variables
columns_to_update = [
    "delayed_recall_raw",  
    "sem_fluency_raw",  
    "tmt_b_raw",  
    "inverse_digits_raw",  
    "inm_recall_total_raw", 
]
np_tp2_df = add_prefix_to_selected_columns(np_tp2_df, columns_to_update)

# Rename the ravlt_total column
np_tp2_df = np_tp2_df.rename(columns={"w2_inm_recall_total_raw": "w2_ravlt_total"})

np_tp2_df[
    [
        "id",
        "w2_ravlt_total",
        "w2_sem_fluency_raw",
        "w2_tmt_b_raw",
        "w2_inverse_digits_raw",
    ]
].sample(5)

In [ ]:
# Merge dfs by participant ID for neuropsych tp2 data and filtered df
filtered_df_tp1 = pd.merge(filtered_df, np_tp2_df, on="id", how="outer")
filtered_df = pd.merge(filtered_df, np_tp2_df, on="id", how="inner")

# Print the number of participants after filtering
print(f"Number of participants: {len(filtered_df)}")
print(f"Number of participants with tp1 data (not filtered for missing data): {len(filtered_df_tp1)}")

filtered_df[["id", "w1_age"]].head(5)

In [ ]:
# Calculate ages at tp2 assessment

# 'Q1 GNRL Age' is DOB and 'nps_date' is data collection date
filtered_df["w2_age"] = filtered_df.apply(lambda row: calculate_age(row["Q1 GNRL Age"], row["nps_date"]), axis=1)

# Only compute w2_age when a timepoint 2 visit exists
filtered_df_tp1["w2_age"] = filtered_df_tp1.apply(
    lambda row: calculate_age(row["Q1 GNRL Age"], row["nps_date"])
    if pd.notna(row["nps_date"]) and pd.notna(row["Q1 GNRL Age"])
    else pd.NA,
    axis=1,
)

# Manually check that everything is running correctly
filtered_df[["id", "w2_age", "Q1 GNRL Age", "nps_date"]].sample(5)

In [ ]:
# Drop participants with missing neuropsych data and -1 values in the specific columns at TP1 ONLY

# Get the number of rows before dropping participants
rows_before = filtered_df_tp1.shape[0]

columns = [
    "w1_age", 
    "YoE",  
    "w1_delayed_recall_raw",  
    "w1_sem_fluency_raw",  
    "w1_tmt_b_raw",  
    "w1_inverse_digits_raw",  
    "w1_ravlt_total",  
]

filtered_df_tp1.dropna(subset=columns, inplace=True)

# Drop participants with -1 values in specific columns
for col in columns:
    filtered_df_tp1 = filtered_df_tp1[filtered_df_tp1[col] != -1]

# Get the number of rows after dropping participants
rows_after = filtered_df_tp1.shape[0]

# Calculate the number of participants dropped
participants_dropped = rows_before - rows_after
participants_dropped

In [ ]:
# Drop participants with missing neuropsych data and -1 values in the specific columns

# Get the number of rows before dropping participants
rows_before = filtered_df.shape[0]

columns = [
    "w1_age",                  # age tp1
    "w2_age",                  # age tp2
    "YoE",                     # years of education
    "w1_ravlt_total",          # RAVLT total tp1
    "w1_delayed_recall_raw",   # RAVLT delayed tp1
    "w1_sem_fluency_raw",      # semantic fluency tp1
    "w1_tmt_b_raw",            # TMT-B tp1
    "w1_inverse_digits_raw",   # digit span backward tp1
    "w2_ravlt_total",          # RAVLT total tp2
    "w2_delayed_recall_raw",   # RAVLT delayed tp2
    "w2_sem_fluency_raw",      # semantic fluency tp2
    "w2_tmt_b_raw",            # TMT-B tp2
    "w2_inverse_digits_raw",   # digit span backward tp2
]

# Drop participants with missing neuropsych data
filtered_df.dropna(subset=columns, inplace=True)

# Drop participants with -1 values in specific columns
for col in columns:
    filtered_df = filtered_df[filtered_df[col] != -1]

# Get the number of rows after dropping participants
rows_after = filtered_df.shape[0]

# Calculate the number of participants dropped
participants_dropped = rows_before - rows_after
participants_dropped

In [ ]:
# Drop people with identical ages at tp1 and tp2
def drop_identical_age_pairs(filtered_df):
    """Drops rows from the DataFrame where the ages at both timepoints are identical.

    Args:
        filtered_df: DataFrame.
    """
    # Filter out rows where the ages are identical
    filtered_data = filtered_df[filtered_df['w1_age'] != filtered_df['w2_age']]

    print(f"Dropped {len(filtered_df) - len(filtered_data)} participants with identical ages at tp1 and tp2.")
    return filtered_data

filtered_df = drop_identical_age_pairs(filtered_df)

In [ ]:
# Rename the sex column
filtered_df = filtered_df.rename(columns={"Q1 GNRL Sex": "sex"})
filtered_df_tp1 = filtered_df_tp1.rename(columns={"Q1 GNRL Sex": "sex"})

# Replace 1 with 'male' and 2 with 'female' in the sex column
filtered_df["sex"] = filtered_df["sex"].replace({1: "male", 2: "female"})
filtered_df_tp1["sex"] = filtered_df_tp1["sex"].replace({1: "male", 2: "female"})

In [ ]:
# Filter df to only include needed columns
columns_to_keep = [
    "id",
    "YoE",
    "sex",
    "w1_age",
    "w1_delayed_recall_raw",
    "w1_tmt_b_raw",
    "w1_sem_fluency_raw",
    "w1_inverse_digits_raw",
    "w1_ravlt_total",
    "w2_age",
    "w2_delayed_recall_raw",
    "w2_sem_fluency_raw",
    "w2_tmt_b_raw",
    "w2_inverse_digits_raw",
    "w2_ravlt_total",
]

filtered_df = filtered_df[columns_to_keep]
filtered_df_tp1 = filtered_df_tp1[columns_to_keep]

In [ ]:
# Drop participants with invalid data according to the the NPS notes
      
# Participants with invalid TMT data
invalid_tmt_tp1 = [64706]
invalid_tmt_tp2 = [60889, 151058]

# Participants with invalid RAVLT data
invalid_ravlt_tp1 = [128826]
invalid_ravlt_tp2 = [127248, 151058, 152027]

# Drop invalid participants from data
filtered_df.loc[filtered_df["id"].isin(invalid_tmt_tp1), ["w1_tmt_b_raw"]] = pd.NA
filtered_df.loc[filtered_df["id"].isin(invalid_tmt_tp2), ["w2_tmt_b_raw"]] = pd.NA
filtered_df.loc[filtered_df["id"].isin(invalid_ravlt_tp1), ["w1_delayed_recall_raw", "w1_ravlt_total"]] = pd.NA
filtered_df.loc[filtered_df["id"].isin(invalid_ravlt_tp2), ["w2_delayed_recall_raw", "w2_ravlt_total"]] = pd.NA

# Apply the same invalid-data masks to the TP1-only dataset
filtered_df_tp1.loc[filtered_df_tp1["id"].isin(invalid_tmt_tp1), ["w1_tmt_b_raw"]] = pd.NA
filtered_df_tp1.loc[filtered_df_tp1["id"].isin(invalid_tmt_tp2), ["w2_tmt_b_raw"]] = pd.NA
filtered_df_tp1.loc[filtered_df_tp1["id"].isin(invalid_ravlt_tp1), ["w1_delayed_recall_raw", "w1_ravlt_total"]] = pd.NA
filtered_df_tp1.loc[filtered_df_tp1["id"].isin(invalid_ravlt_tp2), ["w2_delayed_recall_raw", "w2_ravlt_total"]] = pd.NA

# Manually check that everything is running correctly
ids = [64706, 60889, 151058, 128826, 127248, 152027]
cols = ["id", "w1_tmt_b_raw", "w2_tmt_b_raw", "w1_delayed_recall_raw", "w2_delayed_recall_raw"]
df_sub = filtered_df.loc[filtered_df["id"].isin(ids), cols].copy()
df_sub["id"] = pd.Categorical(df_sub["id"], categories=ids, ordered=True)
df_sub.sort_values("id", kind="stable")

In [ ]:
# Plot the data to make sure there are no other people who need to be dropped
vars = [
        "w1_delayed_recall_raw",
        "w2_delayed_recall_raw",
        "w1_ravlt_total",
        "w2_ravlt_total",
        "w1_tmt_b_raw",
        "w2_tmt_b_raw",
        "w1_sem_fluency_raw",
        "w2_sem_fluency_raw",
        "w1_inverse_digits_raw",
        "w2_inverse_digits_raw",
    ]

step = 4  # Label every 5th bar

for var in vars:
    plt.figure(figsize=(16, 5))  
    ax = sns.countplot(data=filtered_df, x=var)
    plt.title(f'Bar Distribution of {var}')
    plt.xlabel(var)
    plt.ylabel('Count')

    for i, p in enumerate(ax.patches):
        if i % step == 0:
            count = int(round(p.get_height(), 0))
            ax.annotate(str(count),
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, color='black')
    plt.show()

In [ ]:
# Look at data for available longitudinal analysis
participants = filtered_df["w1_age"].count()
mean_age = filtered_df["w1_age"].mean()
std_dev_age = filtered_df["w1_age"].std()
min_age = filtered_df["w1_age"].min()
max_age = filtered_df["w1_age"].max()

print(f"Number of participants: {participants}")
print(f"Mean age: {mean_age:.2f}")
print(f"SD age: {std_dev_age:.2f}")
print(f"Range age: {min_age:.2f} - {max_age:.2f}")

In [ ]:
# Look at data for available tp1 analysis
participants = filtered_df_tp1["w1_age"].count()
mean_age = filtered_df_tp1["w1_age"].mean()
std_dev_age = filtered_df_tp1["w1_age"].std()
min_age = filtered_df_tp1["w1_age"].min()
max_age = filtered_df_tp1["w1_age"].max()

print(f"Number of participants: {participants}")
print(f"Mean age: {mean_age:.2f}")
print(f"SD age: {std_dev_age:.2f}")
print(f"Range age: {min_age:.2f} - {max_age:.2f}")

In [ ]:
# Export this df to a csv to use for future analysis
filtered_df.to_csv("/Users/rachelmorse/Documents/2023:2024/Data/Exported data/clean_bbhi.csv", index=False)
filtered_df_tp1.to_csv("/Users/rachelmorse/Documents/2023:2024/Data/Exported data/clean_bbhi_tp1.csv", index=False)